# Silver: inventory snapshots
- negative stock -> quarantine
- **impossible** stock -> quarantine, judged per item against its own last 28 clean days: `on_hand > 5 x max + 100` (after >= 7 days of history)
- history comes only from clean Silver rows, so a bad value never distorts the baseline

In [0]:
%run ./_common

In [0]:
NB = "21_silver_inventory"
SRC, TGT, QUAR = f"{CAT}.bronze.inventory", f"{CAT}.silver.inventory", f"{CAT}.silver.inventory_quarantine"
CP = f"{CKPT}/silver_inventory"
LOOKBACK, MIN_HISTORY, SPIKE_FACTOR, SPIKE_PAD = 28, 7, 5, 100
KEYS = ["snapshot_date", "store_id", "item_id"]

spark.sql(f"""CREATE TABLE IF NOT EXISTS {TGT} (
  snapshot_date DATE, store_id STRING, item_id STRING, on_hand INT,
  _source_file STRING, _ingested_at TIMESTAMP, _silver_at TIMESTAMP) CLUSTER BY (snapshot_date, store_id)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {QUAR} (
  snapshot_date DATE, store_id STRING, item_id STRING, on_hand INT, prev_max_28d INT, prev_days BIGINT,
  _rescued_data STRING, _source_file STRING, _ingested_at TIMESTAMP, _dq_reason STRING, _quarantined_at TIMESTAMP)""")

def process(batch_df, batch_id):
    s = batch_df.sparkSession
    new = batch_df.select(F.to_date("snapshot_date").alias("snapshot_date"), "store_id", "item_id",
                          F.col("on_hand").cast("int").alias("on_hand"),
                          "_rescued_data", "_source_file", "_ingested_at")
    rng = new.agg(F.min("snapshot_date").alias("lo"), F.max("snapshot_date").alias("hi")).first()
    if rng["lo"] is None:
        return
    hist = (s.table(TGT)
              .where(F.col("snapshot_date").between(F.date_sub(F.lit(rng["lo"]), LOOKBACK), F.lit(rng["hi"])))
              .select("snapshot_date", "store_id", "item_id", "on_hand",
                      F.lit(None).cast("string").alias("_rescued_data"), "_source_file", "_ingested_at"))
    new = new.join(hist.select(*KEYS), KEYS, "left_anti")            # retried batch -> skip rows already in Silver
    w = Window.partitionBy("store_id", "item_id").orderBy("snapshot_date").rowsBetween(-LOOKBACK, -1)
    scored = (new.withColumn("_is_new", F.lit(True))
                .unionByName(hist.withColumn("_is_new", F.lit(False)))
                .withColumn("prev_max_28d", F.max("on_hand").over(w))
                .withColumn("prev_days", F.count("on_hand").over(w))
                .filter("_is_new"))
    reason = (F.when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("on_hand").isNull() | F.col("snapshot_date").isNull(), "missing_value")
               .when(F.col("on_hand") < 0, "negative_inventory")
               .when((F.col("prev_days") >= MIN_HISTORY) &
                     (F.col("on_hand") > F.col("prev_max_28d") * SPIKE_FACTOR + SPIKE_PAD), "impossible_inventory"))
    scored = scored.withColumn("_dq_reason", reason)
    (scored.filter("_dq_reason IS NOT NULL").withColumn("_quarantined_at", F.current_timestamp())
        .select(*s.table(QUAR).columns).write.mode("append")
        .option("txnAppId", "silver_inventory_quarantine").option("txnVersion", batch_id)
        .saveAsTable(QUAR))
    good = (scored.filter("_dq_reason IS NULL").withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), " AND ".join(f"t.{k} = s.{k}" for k in KEYS))
        .whenNotMatchedInsertAll().execute())

before = table_count(TGT)
q = (spark.readStream.option("maxBytesPerTrigger", "64m").table(SRC)
       .writeStream.foreachBatch(process)
       .option("checkpointLocation", CP)
       .trigger(availableNow=True).start())
q.awaitTermination()
print(f"{table_count(TGT) - before:,} clean snapshots added (total {table_count(TGT):,}); quarantine {table_count(QUAR):,}")

In [0]:
b, s, qn = table_count(SRC), table_count(TGT), table_count(QUAR)
dq_check("inventory_reconciles_to_bronze", s + qn, s + qn == b, f"{b} (bronze rows)")
dq_check("inventory_quarantine_rate", qn / max(b, 1), qn / max(b, 1) < 0.001, "< 0.1%")
dq_finish(NB)